In [ ]:
import json
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

# Carregar manifestações
with open('manifestacoes.json', 'r', encoding='utf-8') as f:
    dados = json.load(f)

textos = [d['texto'] for d in dados]
ids = [d['id'] for d in dados]
mapa_textos = dict(zip(ids, textos))

# Gerar representações esparsas
vectorizer_bow = CountVectorizer()
X_bow = vectorizer_bow.fit_transform(textos)

vectorizer_tfidf = TfidfVectorizer()
X_tfidf = vectorizer_tfidf.fit_transform(textos)

# Gerar representações densas (Embeddings)
modelo = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
X_emb = modelo.encode(textos)

# Pares para análise
pares = [("M003", "M017"), ("M008", "M022"), ("M008", "M031")]
resultados = []

for id1, id2 in pares:
    idx1, idx2 = ids.index(id1), ids.index(id2)
    
    sim_bow = cosine_similarity(X_bow[idx1], X_bow[idx2])[0][0]
    sim_tfidf = cosine_similarity(X_tfidf[idx1], X_tfidf[idx2])[0][0]
    sim_emb = cosine_similarity([X_emb[idx1]], [X_emb[idx2]])[0][0]
    
    resultados.append({
        "Par": f"{id1} x {id2}",
        "BoW": f"{sim_bow:.4f}",
        "TF-IDF": f"{sim_tfidf:.4f}",
        "Embeddings": f"{sim_emb:.4f}"
    })

# Exibir resultados
df_resultados = pd.DataFrame(resultados)
print(df_resultados.to_markdown(index=False))

# Conclusão para incluir no Notebook:
# As abordagens esparsas (BoW e TF-IDF) limitam-se à correspondência exata do vocabulário, falhando ao detectar 
# similaridade entre denúncias escritas com palavras distintas (ex: "sem médico" e "falta atendimento"). 
# Os Embeddings superam essa limitação ao capturar o contexto semântico das frases, permitindo o agrupamento correto dos temas.